In [3]:
import csv
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
base = "CME"
threshold = 3.5

file_CME = "Input Data/ap2_CME.csv"
file_Dino = "/scratch1/ajain/cell_interactome/scripts/tracking/ap2_no_feats/ap2_no_feats_centroids.csv"


In [3]:
df_CME = pd.read_csv(file_CME, header = None)
M_CME = df_CME.to_numpy()
M_CME[:, 2] += M_CME[:, 1]-1
M_CME[:, 5] = 169 - M_CME[:, 5]

df_Dino = pd.read_csv(file_Dino)
display(df_Dino)
#df_Dino = df_Dino.drop("max_intensity", axis = 1)
M_Dino = df_Dino.to_numpy()
M_Dino[:, 1:3] += 1

if base == "CME":
    comp_list = np.zeros((len(M_CME),19), dtype=object)
    M_x = M_CME
    M_y = M_Dino
    sec = "Dino"
if base == "Dino":
    comp_list = np.zeros((len(M_Dino),19), dtype=object)
    M_x = M_Dino
    M_y = M_CME
    sec = "CME"

,timepoint,particle_id,x,y,z
0,0,0,93.940050,342.437350,498.42360
1,0,1,0.000000,14.555616,261.02164
2,0,2,0.482249,30.700237,259.63040
3,0,3,1.893685,63.177810,261.31150
4,0,4,1.525761,74.530205,264.57890
...,...,...,...,...,...
196694,98,1478,166.079960,694.838870,518.17840
196695,98,1479,166.169270,701.065900,571.93066
196696,98,1480,166.999820,702.999200,536.58430
196697,98,1481,166.999500,702.997860,548.68300


In [4]:
t_vec = defaultdict(list)
multi_match_list = []
for vec_y in M_y:
    t_vec[vec_y[2]].append(vec_y)

for t_val in t_vec:
    t_vec[t_val] = np.array(t_vec[t_val])

for i, vec_x in enumerate(M_x):
    t_val = vec_x[2]
    y_group = t_vec[t_val]
    diffs = y_group[:, 3:6] - vec_x[3:6]
    dists = np.linalg.norm(diffs, axis=1)
    min_idx = np.argmin(dists)
    min_dist = dists[min_idx]
    best_vec_y = y_group[min_idx]
    comp_list[i, 0:8] = vec_x
    comp_list[i, 8:16] = best_vec_y
    comp_list[i, 16] = min_dist

    below = np.where(dists < threshold)[0]
    if len(below) > 0:
        y_ids = ",".join(str(int(x)) for x in y_group[below, 0])
        dists_str = ",".join(f"{dists[j]:.2f}" for j in below)
    else:
        y_ids = ""
        dists_str = ""
    comp_list[i, 17] = y_ids
    comp_list[i, 18] = dists_str

multi_match_array = np.array(multi_match_list, dtype=object)

ValueError: operands could not be broadcast together with shapes (2008,2) (3,) 

In [ ]:
df_comp = pd.DataFrame(comp_list, columns = [f"ID ({base})",f"t_start ({base})", "t", f"x ({base})", f"y ({base})", 
                                             f"z ({base})", f"FI ({base})", f"Track Length ({base})", f"ID ({sec})", 
                                             f"t_start ({sec})", "t_ig", f"x ({sec})", f"y ({sec})", f"z ({sec})", 
                                             f"FI ({sec})", f"Track Length ({sec})", "Distance", f"Multi ID ({sec})", f"Multi Distance ({sec})"])

df_comp = df_comp.drop("t_ig", axis=1)
df_comp = df_comp[[f"ID ({base})",f"ID ({sec})", f"x ({base})", f"y ({base})", 
                   f"z ({base})", f"x ({sec})", f"y ({sec})", f"z ({sec})", "t",
                   f"t_start ({base})", f"t_start ({sec})",f"FI ({base})", 
                   f"FI ({sec})",f"Track Length ({base})", f"Track Length ({sec})", "Distance",f"Multi ID ({sec})", f"Multi Distance ({sec})"]] 
df_sorted = df_comp.sort_values(by=[f"ID ({base})","t"], ascending=[True, True])

In [ ]:
df_sorted.to_csv(f"{base} Base Output/Comparison.csv", index=False)

In [ ]:
df_sorted

In [1]:
IDs = [491,497,499,519,524,528,541,554,556,557,560,573,574,597,598,623,661,670,674,683,685,711,716,718,722,726,739,789,805,838,845,864,874,884,889,893,932,939,948,1006,1065,1084,1122,1125,1130,1150,1200,1212,1264,1281,1284,1291,1322,1330,1336,1282]

filtered = df_sorted[df_sorted["ID (CME)"].isin(IDs)]
display(filtered)

NameError: name 'df_sorted' is not defined

In [27]:
a = pd.read_csv("Input Data/detections_Dino.csv", names=["timepoint","ID","z","y","x", "FI_ig", "FI"],skiprows=1)
a = a.drop("FI_ig",axis=1)
a["timepoint"] = a["timepoint"] +1
a["x"] = a["x"] +5
a.to_csv("detections_test.csv",index=False)

In [28]:
a = pd.read_csv("ap2_track.csv", names=["ID", "t0", "t", "x", "y", "z","FI", "TL"])
a["t"]=a["t"]+a["t0"]-1
a["z"]= 169-a["z"]
a.to_csv("CME.csv",index=False)
a

,ID,t0,t,x,y,z,FI,TL
0,1,1,1,316.79,151.61,34.960,175.250,99
1,1,1,2,316.91,151.95,35.180,218.140,99
2,1,1,3,316.99,151.34,35.330,218.740,99
3,1,1,4,317.09,152.22,35.240,264.620,99
4,1,1,5,316.93,151.54,35.000,247.410,99
...,...,...,...,...,...,...,...,...
77853,6297,98,99,516.34,265.71,146.049,31.553,2
77854,6298,98,98,548.65,294.81,151.196,61.880,2
77855,6298,98,99,548.09,294.26,149.239,59.079,2
77856,6299,98,98,509.90,300.23,153.892,64.359,2
